# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [ ]:
!pip install -q --upgrade pypdf gradio openai python-dotenv

In [ ]:
!pip install -q --upgrade gradio

### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [ ]:
# Paso 1 — Extracción de texto del PDF
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    """Extrae todo el texto de un PDF como texto plano."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"   # 'or ""' evita None en páginas sin texto
    return text

PDF_PATH = "attention_is_all_you_need.pdf"
document_text = extract_text_from_pdf(PDF_PATH)

print("Caracteres extraídos:", len(document_text))
print("--- primeros 500 caracteres ---")
print(document_text[:500])

Caracteres extraídos: 39600
--- primeros 500 caracteres ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [ ]:
# Paso 2 — Cliente de NVIDIA NIM (API compatible con OpenAI)
import os
from google.colab import userdata
from openai import OpenAI
os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")

# Verifica que la key exista SIN imprimirla
assert os.getenv("NVIDIA_API_KEY"), "Falta NVIDIA_API_KEY en el .env"

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
)

MODELO = "meta/llama-3.2-90b-vision-instruct"
print("Cliente NVIDIA inicializado. Modelo:", MODELO)

Cliente NVIDIA inicializado. Modelo: meta/llama-3.2-90b-vision-instruct


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [ ]:
# Paso 3 — Función de chat fundamentada en el documento

def build_system_prompt(document_text):
    """System prompt que inyecta el documento y obliga a responder solo con él."""
    return f"""Eres un asistente experto en el paper "Attention Is All You Need" (Vaswani et al., 2017).

Responde ÚNICAMENTE con base en el documento delimitado abajo. No uses conocimiento externo.
Si la respuesta no está en el documento, responde exactamente:
"No encuentro esa información en el documento."

===== INICIO DEL DOCUMENTO =====
{document_text}
===== FIN DEL DOCUMENTO =====
"""

def history_to_messages(history):
    """Convierte el historial de Gradio a mensajes con roles user/assistant.
    Soporta el formato moderno ('messages': lista de dicts) y el antiguo (tuplas)."""
    messages = []
    for turn in history or []:
        if isinstance(turn, dict):                      # formato 'messages'
            if turn.get("content"):
                messages.append({"role": turn["role"], "content": turn["content"]})
        else:                                           # formato antiguo [user, assistant]
            user_msg, assistant_msg = turn
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})
    return messages

def chat_con_documento(message, history, document_text):
    """Responde en streaming usando el documento como contexto."""
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages += history_to_messages(history)
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    partial = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            partial += delta
            yield partial   # entrega progresiva (streaming)

## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [ ]:
# Paso 4 — Interfaz Gradio
import gradio as gr

# 1-2. Extrae el texto y reporta tamaño
document_text = extract_text_from_pdf(PDF_PATH)
n_pages = len(PdfReader(PDF_PATH).pages)
print(f"Páginas: {n_pages} | Caracteres: {len(document_text)}")

# 3. ChatInterface con el documento enviado por un Textbox oculto
demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="📄 Asistente del paper 'Attention Is All You Need'",
    description="Preguntas respondidas EXCLUSIVAMENTE con el contenido del paper (Vaswani et al., 2017).",
    additional_inputs=[
        gr.Textbox(value=document_text, visible=False),   # documento oculto
    ],
    examples=[
        ["¿Cuál es la arquitectura principal propuesta en el paper?"],
        ["¿Qué es el mecanismo de atención (self-attention)?"],
        ["¿Cuántas capas tiene el encoder del modelo base?"],
    ],
)

import gradio as gr

gr.close_all()                 # libera los servidores/puertos de lanzamientos anteriores
demo.launch(share=True, show_error=True)

Páginas: 15 | Caracteres: 39600
Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://279a69e2b06d4a3e47.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Paso 5: Prueba y reflexión

Una vez que la interfaz esté funcionando, prueba estas preguntas:

1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*

Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*

Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?


In [ ]:
# ¿La key funciona? ¿Qué modelos tengo disponibles?
try:
    modelos = client.models.list()
    print("✓ La key es válida. Modelos disponibles (primeros 30):")
    for m in list(modelos)[:30]:
        print(" -", m.id)
except Exception as e:
    print("✗ Error:", e)

✓ La key es válida. Modelos disponibles (primeros 30):
 - 01-ai/yi-large
 - adept/fuyu-8b
 - ai21labs/jamba-1.5-large-instruct
 - aisingapore/sea-lion-7b-instruct
 - bigcode/starcoder2-15b
 - databricks/dbrx-instruct
 - deepseek-ai/deepseek-coder-6.7b-instruct
 - deepseek-ai/deepseek-v4.1-flash
 - google/codegemma-1.1-7b
 - google/codegemma-7b
 - google/deplot
 - google/diffusiongemma-26b-a4b-it
 - google/gemma-2b
 - google/gemma-3-12b-it
 - google/gemma-3-4b-it
 - google/gemma-4-31b-it
 - google/recurrentgemma-2b
 - ibm/granite-3.0-3b-a800m-instruct
 - ibm/granite-3.0-8b-instruct
 - ibm/granite-34b-code-instruct
 - ibm/granite-8b-code-instruct
 - meta/codellama-70b
 - meta/llama-3.2-11b-vision-instruct
 - meta/llama-3.2-90b-vision-instruct
 - meta/llama-guard-4-12b
 - meta/llama2-70b
 - meta/muse-glimmer-30b
 - microsoft/kosmos-2
 - microsoft/phi-3-vision-128k-instruct
 - microsoft/phi-3.5-moe-instruct


### Observación de la prueba (Paso 5)

- Preguntas **1-5** (arquitectura, atención, 6 capas del encoder, autores, BLEU 28.4 en WMT14 EN-DE): el modelo debe responderlas correctamente **citando el documento**, porque toda esa información está en el texto inyectado.
- Pregunta **6 (GPT-4, la trampa):** no aparece en el paper (2017). Con el system prompt bien construido, el modelo debe responder *"No encuentro esa información en el documento."* en lugar de usar su conocimiento general. Si responde sobre GPT-4, significa que **la instrucción de fundamentación no está siendo respetada** — señal de que el prompt necesita ser más estricto.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?
2. ¿Por qué este ejercicio todavía no implementa RAG?
3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
   conocimiento previo del modelo?
4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)

### Respuestas — Reflexión final

1. **Limitación de enviar el documento completo:** cada pregunta re-envía TODO el texto (~9.900 tokens para este paper) en el contexto. Es **caro** (se paga por token en cada llamada), **lento**, y **no escala**: un documento grande o varios documentos superan la ventana de contexto del modelo. También diluye la atención entre mucho texto irrelevante.

2. **Por qué esto todavía NO es RAG:** aquí no hay **recuperación** — se mete el documento entero sin seleccionar nada. RAG primero *divide* el documento en fragmentos, *recupera* solo los más relevantes a la pregunta (vía embeddings + búsqueda vectorial) y *solo esos* van al prompt. Este ejercicio es el "antes" que motiva RAG.

3. **Cómo comprobar que la respuesta viene del documento y no del conocimiento previo:** (a) la **pregunta trampa** (GPT-4) — si se niega, respeta el documento; (b) pedir que **cite la página/sección** de donde saca cada dato; (c) preguntar algo que contradiga el conocimiento general pero esté en el doc, y ver cuál gana.

4. **Riesgos de insertar documentos de usuarios en el prompt:** **prompt injection** (el documento puede contener instrucciones maliciosas tipo "ignora tus reglas y haz X"), **fuga de datos** (información sensible enviada a un tercero/servidor del proveedor), y **costo/DoS** (un documento enorme dispara tokens y gasto). Nunca se debe confiar ciegamente en el contenido inyectado.